# Лабораторная работа 5.
**Временные ряды**

Задачи:
* Предварительный анализ и визуализация данных.
Загрузить данные и построить графики временных рядов занятости
Определить признаки сезонности, тренда и возможные аномалии

* Проверка стационарности.
Выполнить тесты  для проверки стационарности ряда
При необходимости выполнить преобразования
* Построение модели прогнозирования.
Построить и обучить модель прогнозирования временных рядов
Сделать прогноз на несколько периодов  
Оценить точность прогноза с помощью метрик MAE, RMSE.
* Анализ и интерпретация результатов.
Визуализировать полученный прогноз вместе с исходным рядом
Интерпретировать выявленные сезонные и трендовые компоненты
Обсудить влияние сезонности на прогноз


In [ ]:
!gdown 1T_44FkDI_bNHUIROJWPgTcfkqAfMKttD

Downloading...
From: https://drive.google.com/uc?id=1T_44FkDI_bNHUIROJWPgTcfkqAfMKttD
To: /content/lab5.csv
100% 5.60k/5.60k [00:00<00:00, 15.1MB/s]


In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

In [ ]:
data = pd.read_csv("/content/lab5.csv")
data.Date = pd.to_datetime(data.Date)
data = data.set_index("Date")
data = data.asfreq("MS")
display(data.head())
display(data.describe().T)
display(data.info())

,Employees
Date,
1990-01-01,1064.5
1990-02-01,1074.5
1990-03-01,1090.0
1990-04-01,1097.4
1990-05-01,1108.7


,count,mean,std,min,25%,50%,75%,max
Employees,348.0,1452.506897,256.604914,1064.5,1238.05,1436.2,1586.3,2022.1


<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 348 entries, 1990-01-01 to 2018-12-01
Freq: MS
Data columns (total 1 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Employees  348 non-null    float64
dtypes: float64(1)
memory usage: 5.4 KB


None

# 1 Предварительный анализ

In [ ]:
fig = px.line(data, x=data.index, y="Employees")
fig.show()

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose
from plotly.subplots import make_subplots
import plotly.graph_objects as go

In [ ]:
decomposition = seasonal_decompose(data['Employees'], model='additive')

In [ ]:
fig = make_subplots(
    rows=3,
    cols=1,
    row_titles="Сезонность Тренд Остатки".split()
)
fig.add_trace(
    go.Scatter(
        x=decomposition.seasonal.index,
        y=decomposition.seasonal,
    ),
    row=1, col=1,

)
fig.add_trace(
    go.Scatter(
        x=decomposition.trend.index,
        y=decomposition.trend,
    ),
    row=2, col=1,

)
fig.add_trace(
    go.Scatter(
        x=decomposition.resid.index,
        y=decomposition.resid,
    ),
    row=3, col=1,

)
fig.update_layout(showlegend=False)
fig.show()

Итак, данные у нас по количеству работников в гостиничном бизнесе в шатет Калифорния. Штат достаточно туристический: парк Йоссемити, мост Золотые Ворота, Алькатрас. Наблюдается явная сезонность в данных, когда в летние месяца туристический поток достигает пиковых значений необходимо увеличивать штат, а в зимние месяца все наоборот.

Также наблюдается уверенный восходящий тренд, среднне количество сотрудников растет из года в год

Аномальный 2008 год, связан с ипотечным кризисом, многие потеряли работу, и решили повременить с отпуском это каскадом отразилось на гостиничном бизнесе

# 2 Проверка стационарности


Критерий Дики-Фуллера

In [ ]:
import statsmodels.api as sm

In [ ]:
test = sm.tsa.stattools.adfuller(data)
print(f"Adf: {test[0]}")
print(f"Pvalue: {test[1]}")
print(f"Critical Values: {test[4]}")
print("Стационарен: ", test[1] > 0.05)

Adf: 0.9012844235569791
Pvalue: 0.9931070655289933
Critical Values: {'1%': np.float64(-3.4503224123605194), '5%': np.float64(-2.870338478726661), '10%': np.float64(-2.571457612488522)}
Стационарен:  True


Критерий KPSS

In [ ]:
kpss = sm.tsa.stattools.kpss(data)
print(f"KPSS: {kpss[0]}")
print(f"Pvalue: {kpss[1]}")
print(f"Critical Values: {kpss}")
print("Стационарен: ", kpss[1] <= 0.05)

KPSS: 2.811347117430525
Pvalue: 0.01
Critical Values: (np.float64(2.811347117430525), np.float64(0.01), 11, {'10%': 0.347, '5%': 0.463, '2.5%': 0.574, '1%': 0.739})
Стационарен:  True


/tmp/ipython-input-2350993200.py:1: InterpolationWarning:

The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.




# 3 Построение модели прогнозирования

In [ ]:
train, test = data.iloc[:-12], data.iloc[-12:]

In [ ]:
def draw(data, forecats, pred):
  fig = go.Figure()

  fig.add_scatter(x=data.index, y=data.Employees, name="Train", line=dict(color='blue'))
  fig.add_scatter(x=forecast.index, y=forecast, name="Test", line=dict(color='red'))
  fig.add_scatter(x=pred.index[10:], y=pred.values[10:], name="Predict", line=dict(color='green'))
  fig.update_xaxes(range=[data.index[-80], pd.to_datetime(pred.index[-1])])
  fig.show()

In [ ]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

## Модель Хольта
Экспоненциальное сглаживание

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

In [ ]:
es = ExponentialSmoothing(train, seasonal="additive", trend="additive", seasonal_periods=12)
es_fit = es.fit()
print(es_fit.summary())

                       ExponentialSmoothing Model Results                       
Dep. Variable:                Employees   No. Observations:                  336
Model:             ExponentialSmoothing   SSE                           9709.527
Optimized:                         True   AIC                           1162.221
Trend:                         Additive   BIC                           1223.294
Seasonal:                      Additive   AICC                          1164.378
Seasonal Periods:                    12   Date:                 Fri, 31 Oct 2025
Box-Cox:                          False   Time:                         08:37:23
Box-Cox Coeff.:                    None                                         
                          coeff                 code              optimized      
---------------------------------------------------------------------------------
smoothing_level               0.8187027                alpha                 True
smoothing_trend          

In [ ]:
forecast = es_fit.forecast(steps=12)

rmse = root_mean_squared_error(test, forecast)
mae = mean_absolute_error(test, forecast)

print(f'MAE: {mae}')
print(f'RMSE: {rmse}')

pred = es_fit.forecast(36)
draw(data, forecast, pred)

MAE: 4.268958812573298
RMSE: 7.019620840418266


## Авторегрессия SARIMAX

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

In [ ]:
sarimax = SARIMAX(train, seasonal_order=(1, 0, 1, 12))
sarimax_fit = sarimax.fit()
print(sarimax_fit.summary())

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning:

Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:997: UserWarning:

Non-stationary starting seasonal autoregressive Using zeros as starting parameters.



                                      SARIMAX Results                                       
Dep. Variable:                            Employees   No. Observations:                  336
Model:             SARIMAX(1, 0, 0)x(1, 0, [1], 12)   Log Likelihood               -1082.823
Date:                              Fri, 31 Oct 2025   AIC                           2173.646
Time:                                      08:37:24   BIC                           2188.914
Sample:                                  01-01-1990   HQIC                          2179.732
                                       - 12-01-2017                                         
Covariance Type:                                opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1          0.9986      0.001    755.038      0.000       0.996       1.001
ar.S.L12       0.99

In [ ]:
forecast = sarimax_fit.forecast(steps=12)

rmse = root_mean_squared_error(test, forecast)
mae = mean_absolute_error(test, forecast)

print(f'MAE: {mae}')
print(f'RMSE: {rmse}')

pred = sarimax_fit.forecast(36)
draw(data, forecast, pred)

MAE: 10.457870082987048
RMSE: 13.356291319575758


В ходе лабораторной работы был проведен EDA временного ряда. Выявлены признаки сезонности и тренда, а также влияние 2008 года.
Были обучены 2 модели для предсказания временного ряда: SARIMAX, HoltWinter.

Из двух моделей лучшее предсказание дала модель экспоненциального сглаживания Хольта Винтера. Обе модели учитывают сезонность - к лету количество сотрудников растет, к зиме падает, и поддерживают восходящий тренд
